In [1]:
import json
from pathlib import Path

import numpy as np
import rasterio
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset

DATA = Path("../data/extracted")
BASES = {
    "amazon": DATA / "amazon" / "AMAZON",
    "atlantic": DATA / "atlantic" / "ATLANTIC FOREST",
}
# image folder, mask folder for each split (same names in both biomes)
SPLIT_DIRS = {
    "Training": ("image", "label"),
    "Validation": ("images", "masks"),
    "Test": ("image", "mask"),
}
STATS = json.loads(Path("../data/band_stats.json").read_text())

In [2]:
class ForestDataset(Dataset):
    """One biome, one split. train=True -> random crop + flips/rotations."""

    def __init__(self, biome, split, train=False, crop_size=256):
        img_dir, msk_dir = SPLIT_DIRS[split]
        base = BASES[biome] / split
        self.biome = biome
        self.img_paths = sorted((base / img_dir).glob("*.tif"))   # *.tif only, skips .aux.xml
        self.msk_dir = base / msk_dir
        self.train = train
        self.crop_size = crop_size
        self.scale = STATS["scale"]
        self.mean = np.array(STATS["mean"], dtype=np.float32).reshape(4, 1, 1)
        self.std = np.array(STATS["std"], dtype=np.float32).reshape(4, 1, 1)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, i):
        p = self.img_paths[i]
        with rasterio.open(p) as src:
            img = src.read()                                  # (4, 512, 512) uint16
        with rasterio.open(self.msk_dir / p.name) as src:
            msk = src.read(1)                                 # (512, 512) uint8, 0/1

        # preprocessing: fixed scale -> clip -> standardize (same constants at inference)
        img = np.clip(img.astype(np.float32) / self.scale, 0.0, 1.0)
        img = (img - self.mean) / self.std

        if self.train:
            c = self.crop_size
            top = np.random.randint(0, img.shape[1] - c + 1)
            left = np.random.randint(0, img.shape[2] - c + 1)
            img = img[:, top:top + c, left:left + c]
            msk = msk[top:top + c, left:left + c]
            if np.random.rand() < 0.5:
                img, msk = img[:, :, ::-1], msk[:, ::-1]
            if np.random.rand() < 0.5:
                img, msk = img[:, ::-1, :], msk[::-1, :]
            k = np.random.randint(0, 4)
            img, msk = np.rot90(img, k, axes=(1, 2)), np.rot90(msk, k)

        x = torch.from_numpy(np.ascontiguousarray(img))
        y = torch.from_numpy(np.ascontiguousarray(msk)).float().unsqueeze(0)
        return x, y

In [3]:
train_ds = ConcatDataset([ForestDataset(b, "Training", train=True) for b in BASES])
print("training images:", len(train_ds))

loader = DataLoader(train_ds, batch_size=8, shuffle=True)
x, y = next(iter(loader))
print("x:", x.shape, x.dtype, "| y:", y.shape, y.dtype, "| mask values:", y.unique())
print("per-band mean:", x.mean(dim=(0, 2, 3)))
print("per-band std: ", x.std(dim=(0, 2, 3)))

training images: 984
x: torch.Size([8, 4, 256, 256]) torch.float32 | y: torch.Size([8, 1, 256, 256]) torch.float32 | mask values: tensor([0., 1.])
per-band mean: tensor([ 0.1954,  0.1132,  0.1460, -0.1747])
per-band std:  tensor([1.1097, 1.0116, 1.0001, 0.6648])
